# Notebook 11 — API cost and request audit

This notebook answers: **What did we send, what failed, and what do we know about the cost?** It reads private attempt logs, makes no API calls, and does not change model scores or labels. Replaying a cached answer does not count as another paid request.

On 2 October, the reviewer confirmed $0 charges for the earlier Groq and Hetzner experiments. That is a user-reported balance, not an invoice verification. Missing token usage remains unknown even when the account incurred no charge.


In [ ]:
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "src"))
from notebooks.tools.costs import audit_attempts
from notebooks.tools import show_table

report = audit_attempts(root / "reports")
show_table(
    ["Experiment folder", "Model", "Attempts", "Missing usage", "Interrupted", "HTTP 429"],
    [
        [
            g["scope"],
            g["model"],
            g["attempts"],
            g["missing_usage"],
            g["missing_completion"],
            g["http_429"],
        ]
        for g in report["groups"]
    ],
)


## 1. Read the counts

An **attempt** is one HTTP send, including a retry. **Missing usage** means the provider did not return both input and output token counts; it does not mean zero tokens. **Interrupted** means there is a start record without a completion record. An HTTP 429 may mean a temporary rate limit or exhausted credits; inspect the saved error classification before retrying.

The first historical audit found 682 attempts, 158 without usage and one without a completion. A later synthetic OpenAI access check adds another attempt. These are the logs we have, not proof that every past call was captured. Old model reports retain original request timings; individual legacy attempt timings and endpoints were not recorded consistently.


In [ ]:
show_table(
    ["Model", "Known input tokens", "Known output tokens"],
    [
        [g["model"], g["known_prompt_tokens"], g["known_completion_tokens"]]
        for g in report["groups"]
    ],
)
for limitation in report["limitations"]:
    print("-", limitation)


## 2. Spending limits before a paid request

`notebooks/tools/costs.py` supplies a persistent reservation ledger. Before sending a request, its adapter must reserve a conservative maximum charge. Retries reserve again. A crash does not erase an earlier reservation. Limits are $50 for the project, $10 for research within that total, $5 for one run, and $1 for one demonstration across runs. Reservations are ceilings, not actual charges; they are not automatically refunded when usage is missing.

The provider adapter must bound billed input, output/reasoning and tool use. A token-only calculation is insufficient for search. The guard is tested and used by the synthetic OpenAI smoke check; it is **not yet integrated into a paid search experiment** or retrofitted into the earlier free-tier runners. Those runners must not be reused for paid calls as they stand.

## 3. Current finding and next action

The GPT-5.4 mini synthetic access test returned HTTP 429 with `credit_balance_exhausted` and `insufficient_quota`. No model output or usage was returned. Its $0.008652 maximum reservation remains in the ledger as a conservative charge allowance; it is not an assertion that this amount was billed. No demonstration titles were sent.

Next, make API credits available, verify access with a synthetic request, and then connect the guard to the bounded search/no-search experiment. Start with GPT-5.4 mini; Astra is not required. Gemini Flash remains a separate possible follow-up, subject to its API access and grounding charges.

**Review:** Check that the table is readable and that unknown costs are clearly distinguished from the confirmed $0 earlier charges. No CSV labeling is needed. DR-24 stays In Progress until the provider integration and cost reconciliation are complete.
